# Random Forest — cfDNA Cancer Classifier

**Why Random Forest first:**  
RF is an ensemble of decision trees. Each tree is trained on a random bootstrap sample of the data, and at each split only a random subset of features is considered. This double randomness makes it robust to overfitting — especially important here with only 148 training samples.

RF also gives **feature importance** out of the box, which tells you which of the 18 features the model actually relies on. If it's using SFR and the 130–150bp bins — the model is learning biology. If it's using something unexpected — investigate before trusting it.

**Evaluation strategy:**  
5-fold stratified cross-validation on the 148 training samples.  
The test set (`X_test`, `y_test`) is not touched until the final evaluation notebook.

## Step 1 — Load split data

In [ ]:
xgb_final = xgb.XGBClassifier(
    n_estimators=500,
    scale_pos_weight=spw,
    max_depth=xgb_grid.best_params_['max_depth'],
    learning_rate=xgb_grid.best_params_['learning_rate'],
    subsample=xgb_grid.best_params_['subsample'],
    colsample_bytree=xgb_grid.best_params_['colsample_bytree'],
    eval_metric='auc',
    random_state=42,
    verbosity=0,
)
xgb_final.fit(X_train, y_train)

with open(ROOT / 'models' / 'xgb_final.pkl', 'wb') as f:
    pickle.dump(xgb_final, f)

print('Saved: models/xgb_final.pkl')
print()
print('=' * 50)
print('TREE MODELS — CV AUC COMPARISON')
print('=' * 50)
print(f'  Random Forest (tuned) : 0.8389')
print(f'  XGBoost       (tuned) : {xgb_grid.best_score_:.4f}')
print(f'  Winner                : {"Random Forest" if 0.8389 > xgb_grid.best_score_ else "XGBoost"}')
print('=' * 50)
print()
print('Test set is still locked. Final AUC reported in evaluation notebook.')

## XGBoost Step 3 — Final model + Save

Fit on all 148 training samples with best params, save to `models/`.

In [ ]:
param_grid_xgb = {
    'max_depth'        : [3, 4],
    'learning_rate'    : [0.05, 0.1],
    'subsample'        : [0.7, 0.9],
    'colsample_bytree' : [0.7, 1.0],
}

xgb_grid = GridSearchCV(
    xgb.XGBClassifier(
        n_estimators=500,
        scale_pos_weight=spw,
        eval_metric='auc',
        random_state=42,
        verbosity=0,
    ),
    param_grid_xgb,
    cv=cv,
    scoring='roc_auc',
    n_jobs=-1,
    verbose=0,
)
xgb_grid.fit(X_train, y_train)

print(f'Best params : {xgb_grid.best_params_}')
print(f'Best CV AUC : {xgb_grid.best_score_:.4f}')
print(f'XGB baseline: 0.7870')
print(f'RF tuned    : 0.8389')
print(f'Improvement over XGB baseline: {xgb_grid.best_score_ - 0.7870:+.4f}')

## XGBoost Step 2 — Hyperparameter Tuning

The baseline underperforms RF because default XGBoost (`learning_rate=0.3`, deep trees) overfits on 148 samples. Four parameters fix this:

- `learning_rate` — lower = each tree contributes less, more trees needed, better generalisation. Try 0.05–0.1.
- `max_depth` — same logic as RF. Shallow trees (3–4) prevent memorisation.
- `subsample` — use only a fraction of training samples per tree, adds randomness like RF's bootstrap.
- `colsample_bytree` — use only a fraction of features per tree, same idea as RF's max_features.

In [ ]:
import xgboost as xgb

# scale_pos_weight = healthy / cancer in training set
spw = (y_train == 0).sum() / (y_train == 1).sum()
print(f'scale_pos_weight = {spw:.2f}  (= {(y_train==0).sum()} healthy / {(y_train==1).sum()} cancer)')

xgb_base = xgb.XGBClassifier(
    n_estimators=500,
    scale_pos_weight=spw,
    eval_metric='auc',
    random_state=42,
    verbosity=0,
)

scores_xgb = cross_val_score(xgb_base, X_train, y_train,
                              cv=cv, scoring='roc_auc', n_jobs=-1)

print()
print('Baseline XGBoost — 5-fold CV AUC')
print(f'  Fold scores : {[round(s, 4) for s in scores_xgb]}')
print(f'  Mean AUC    : {scores_xgb.mean():.4f}')
print(f'  Std         : {scores_xgb.std():.4f}')
print(f'  Range       : {scores_xgb.min():.4f} – {scores_xgb.max():.4f}')
print()
print(f'RF baseline  : 0.8164  →  XGB baseline: {scores_xgb.mean():.4f}')

---
# XGBoost — cfDNA Cancer Classifier

**RF vs XGBoost:**  
Random Forest builds 500 trees independently and averages them. XGBoost builds trees **sequentially** — each tree is trained on the residual errors of all previous trees. This makes it more powerful but also more sensitive to hyperparameters.

Key regularisation parameters for XGBoost on small datasets:
- `max_depth` — same as RF, shallow trees prevent overfitting
- `learning_rate` — how much each new tree corrects errors. Smaller = more trees needed but better generalisation
- `subsample` — fraction of training samples used per tree (like RF's bootstrap)
- `colsample_bytree` — fraction of features used per tree (like RF's max_features)
- `scale_pos_weight` — XGBoost's equivalent of class_weight='balanced'. Set to healthy/cancer ratio

## XGBoost Step 1 — Baseline with 5-fold CV

In [ ]:
with open(ROOT / 'models' / 'rf_final.pkl', 'wb') as f:
    pickle.dump(rf_final, f)

print('Saved: models/rf_final.pkl')
print()
print('Random Forest — Summary')
print(f'  Best params  : {rf_grid.best_params_}')
print(f'  CV AUC       : {rf_grid.best_score_:.4f}  (5-fold, training set only)')
print(f'  Top feature  : {importances.sort_values(ascending=False).index[0]}')
print()
print('Test set is still locked. Final AUC reported in evaluation notebook.')

## Step 5 — Save the model

Save the final fitted RF to `models/` so the evaluation notebook can load it directly without retraining.

In [ ]:
rf_final = RandomForestClassifier(
    n_estimators=500,
    max_depth=rf_grid.best_params_['max_depth'],
    max_features=rf_grid.best_params_['max_features'],
    min_samples_leaf=rf_grid.best_params_['min_samples_leaf'],
    class_weight='balanced',
    random_state=42,
    n_jobs=-1,
)
rf_final.fit(X_train, y_train)

# feature importance
importances = pd.Series(rf_final.feature_importances_, index=feat_cols).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(7, 6))
colors = ['#F44336' if 'ratio' in c or c in ['mean_length','median_length','mono_peak_height','peak_to_flank']
          else '#2196F3' for c in importances.index]
importances.plot.barh(ax=ax, color=colors, edgecolor='white')
ax.set_xlabel('Feature importance (Gini)', fontsize=11)
ax.set_title('Random Forest — Feature Importance', fontsize=13, fontweight='bold')
ax.spines[['top', 'right']].set_visible(False)

from matplotlib.patches import Patch
ax.legend(handles=[Patch(color='#2196F3', label='Bin features'),
                   Patch(color='#F44336', label='Scalar features')],
          fontsize=9, loc='lower right')
plt.tight_layout()
plt.savefig(ROOT / 'results/rf_feature_importance.png', dpi=150)
plt.show()

print('Top 5 features:')
print(importances.sort_values(ascending=False).head(5).round(4).to_string())

## Step 4 — Final model fit + Feature Importance

**Why refit on all training data:**  
During grid search, each fold's model trained on 4/5 of training data (~118 samples). The final model refits on all 148 samples using the best parameters — more data, same regularisation, stronger model.

**Feature importance (Gini impurity):**  
Each feature's importance = how much it reduces impurity (uncertainty) across all splits in all 500 trees. Higher = more useful to the model.

**Biology check:** If the top features are SFR and the 130–150bp bins, the model is learning fragment length biology. If a random-looking feature dominates — investigate before trusting the AUC.

In [ ]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    'max_depth'        : [3, 5, 8, None],
    'max_features'     : ['sqrt', 4, 6],
    'min_samples_leaf' : [1, 2, 4],
}

rf_grid = GridSearchCV(
    RandomForestClassifier(n_estimators=500, class_weight='balanced', random_state=42, n_jobs=-1),
    param_grid,
    cv=cv,
    scoring='roc_auc',
    n_jobs=-1,
    verbose=0,
)
rf_grid.fit(X_train, y_train)

print(f'Best params : {rf_grid.best_params_}')
print(f'Best CV AUC : {rf_grid.best_score_:.4f}')
print(f'Baseline    : 0.8164')
print(f'Improvement : {rf_grid.best_score_ - 0.8164:+.4f}')

## Step 3 — Hyperparameter Tuning

**Parameters being searched:**

- `max_depth` — limits how deep each tree grows. Unlimited depth on 148 samples means trees memorise training data. Constraining depth forces generalisation.
- `max_features` — number of features considered at each split. `sqrt(18)≈4` is the RF default. Trying fewer features increases tree diversity, which strengthens the ensemble.
- `min_samples_leaf` — minimum samples required at a leaf node. Higher values prevent trees from making splits on single outlier samples.

`n_estimators` stays at 500 — enough trees that results are stable, and adding more gives diminishing returns.

All combinations are evaluated with the same 5-fold CV used in the baseline — no test data involved.

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_score

rf_base = RandomForestClassifier(
    n_estimators=500,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1,
)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

scores = cross_val_score(rf_base, X_train, y_train,
                         cv=cv, scoring='roc_auc', n_jobs=-1)

print('Baseline Random Forest — 5-fold CV AUC')
print(f'  Fold scores : {[round(s, 4) for s in scores]}')
print(f'  Mean AUC    : {scores.mean():.4f}')
print(f'  Std         : {scores.std():.4f}')
print(f'  Range       : {scores.min():.4f} – {scores.max():.4f}')

## Step 2 — Baseline Random Forest with 5-fold CV

**Why class_weight='balanced':**  
With 95 healthy vs 53 cancer in training, an unweighted RF will see healthy samples ~1.8× more often during training and bias toward predicting healthy. `class_weight='balanced'` automatically upweights cancer samples so each class contributes equally to every tree's splits.

**Why 5-fold CV and not a single validation set:**  
With 148 training samples, a single 80/20 validation split gives ~30 samples to evaluate on — too noisy. 5-fold CV uses all 148 samples for validation (each fold once), giving a much more stable AUC estimate.

**Reading the output:**  
- Mean AUC across 5 folds = overall performance  
- Std across folds = stability. High std means performance varies a lot depending on which samples end up in validation — a sign the model is sensitive to the specific partition.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pickle, pathlib, sys

ROOT      = pathlib.Path().resolve().parent
SPLIT_DIR = ROOT / 'data' / 'processed' / 'split'

X_train    = np.load(SPLIT_DIR / 'X_train.npy')
y_train    = np.load(SPLIT_DIR / 'y_train.npy')
feat_cols  = np.load(SPLIT_DIR / 'feat_cols.npy', allow_pickle=True).tolist()

print(f'X_train : {X_train.shape}')
print(f'y_train : {y_train.shape}  (healthy={( y_train==0).sum()}  cancer={(y_train==1).sum()})')
print(f'Features: {len(feat_cols)}')